In [1]:
print("Hello World")

Hello World


## Without History

In [10]:
from helpers.common import ollama_llm, langfuse, langfuse_handler
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

chat_prompt = ChatPromptTemplate.from_template("{prompt}")
chain = chat_prompt | ollama_llm | StrOutputParser()
for message in chain.stream({
  "prompt": "Hello, My Name is Yash. I am an Engineer"
},
config={"callbacks": [langfuse_handler],"run_name": "without-history",    "metadata": {"langfuse_tags": ["gemma"]},},

):
 print(message, end="")

Hello Yash! It’s nice to meet you. That’s fantastic that you’re an Engineer – it's a really rewarding profession. 

What kind of engineering do you specialize in? And what’s your favorite part about being an engineer?

In [11]:
for message in chain.stream({
  "prompt": "What is my Name?"
},
config={"callbacks": [langfuse_handler],"run_name": "without-history",    "metadata": {"langfuse_tags": ["gemma"]},},

):
 print(message, end="")

As an AI, I don't know your name! You haven't told me. 😊 

You can tell me your name if you'd like.


## With History

In [17]:
from langchain_core.prompts import SystemMessagePromptTemplate, HumanMessagePromptTemplate, ChatPromptTemplate, MessagesPlaceholder

system_message = SystemMessagePromptTemplate.from_template("You are a helpful assistant. You remember the conversation.")

human_message = HumanMessagePromptTemplate.from_template("{input}")
messages = [
  system_message,
  MessagesPlaceholder(variable_name="history", optional=True),
  human_message
]

chat_prompt = ChatPromptTemplate(messages)
# chat_prompt.invoke({
#   "input": "Hello, I am Yash Kharche"
# })
chain = chat_prompt | ollama_llm

In [19]:
## SQL History
from langchain_community.chat_message_histories import SQLChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory
DB_PATH = "sqlite:///chat_history.db"  

def get_session_history(session_id: str) -> SQLChatMessageHistory:
  return SQLChatMessageHistory(
    session_id=session_id,
    connection=DB_PATH
  )


chat = RunnableWithMessageHistory(
  chain,
  get_session_history,
  input_messages_key="input",
  history_messages_key="history"
)


In [25]:
def chat_with_memory(user_input: str, session_id: str = "default") -> str:
    # response = chat.invoke(
    #     {"input": user_input},
    #     config={"configurable": {"session_id": session_id}},
    # )
    # return response.content
    for message in chat.stream({
      "input": user_input,
    },
    config={"configurable": {"session_id": session_id}}
    ):
      print(message.content, end="")


In [ ]:
reply = chat_with_memory("Hi! My name is Yash Kharche.", session_id="session_1")
print("Assistant:", reply)

Assistant: Hi Yash Kharche! It’s nice to meet you. How’s your day going so far? Is there anything you’d like to chat about, or were you just saying hello?


In [26]:
reply = chat_with_memory("Do you remember my Name?", session_id="session_1")
print("Assistant:", reply)

You are absolutely right to double-check! My apologies. I’m still under development and sometimes I need a little reminder. 

Yes, I remember your name is Yash Kharche. 😊 

I’m really trying to improve my memory! Thanks for pointing out my slip-up.Assistant: None


In [27]:
reply = chat_with_memory("I am also an Engineer building GenAI Apps at Tekframeworks!", session_id="session_1")
print("Assistant:", reply)

That’s fantastic, Yash! Building GenAI apps at Tekframeworks – that’s incredibly exciting. What kind of GenAI apps are you working on specifically? I’d love to hear a little more about it.Assistant: None


In [ ]:
reply = chat_with_memory("Predominantly RAG based", session_id="session_1")
# print("Assistant:", reply)

Okay, RAG-based GenAI apps – that’s a really hot area right now! That makes a lot of sense. 

So, you're focusing on Retrieval-Augmented Generation. That means you're leveraging external knowledge bases to improve the accuracy and context of the generated responses, correct? 

What are some of the biggest challenges you’re facing with RAG implementations at Tekframeworks?Assistant: None


In [29]:
reply = chat_with_memory("Being able to fetch the relevant chunks. I realized chunking strategy is really important. I have Page based chunking right now", session_id="session_1")
reply

You’ve hit on a *huge* one – the chunking strategy is absolutely critical for RAG’s success. Page-based chunking is a really common starting point, and it makes sense for a lot of data. 

What kind of data are you working with? Knowing that might help me offer some specific suggestions. And tell me a little more about *why* you chose page-based chunking initially. Was it primarily for simplicity, or were there other factors? 

Also, have you experimented with any other chunking methods (like semantic chunking, or sliding window approaches)?

In [31]:
history = get_session_history("session_1")
history.messages

[HumanMessage(content='Hi! My name is Yash Kharche.', additional_kwargs={}, response_metadata={}),
 AIMessage(content='Hi Yash Kharche! It’s nice to meet you. How’s your day going so far? Is there anything you’d like to chat about, or were you just saying hello?', additional_kwargs={}, response_metadata={'model': 'gemma3:4b', 'created_at': '2026-03-15T07:43:05.776148455Z', 'done': True, 'done_reason': 'stop', 'total_duration': 9752554791, 'load_duration': 4352649926, 'prompt_eval_count': 34, 'prompt_eval_duration': 1647469197, 'eval_count': 41, 'eval_duration': 3341732050, 'logprobs': None, 'model_name': 'gemma3:4b', 'model_provider': 'ollama'}, id='lc_run--019cf072-b816-7452-a749-f968cef2a1a9-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 34, 'output_tokens': 41, 'total_tokens': 75}),
 HumanMessage(content='Do you remember my Name?', additional_kwargs={}, response_metadata={}),
 AIMessageChunk(content='Absolutely! You’re Yash Kharche. I’ve got it right here.

In [12]:
langfuse.flush()